# 05 · Orchestration

So far LangGraph built the graph for you: `create_agent` is a two-node loop where the **model** decides
every step. Many systems are better as a **workflow**, where **code** decides the steps and the model
fills in the parts that need language. LangGraph's `StateGraph` lets you draw either, or mix them.

**You will learn**

- how to build a graph yourself: a typed state, nodes, edges and conditional edges
- when to use a workflow, an agent, or an agent inside a workflow
- how to build a loop whose limit is enforced in code, not in the prompt
- the research agent's design, which the capstone builds on

In [1]:
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, StateGraph

from agent_contracts.harness import verify_model
from lg_agents.llm import make_llm

verify_model()
llm = make_llm()

## A first graph: routing ShipFast messages

A `StateGraph` needs three things. A **state** schema (a `TypedDict`): the shared record every node reads
and updates. **Nodes**: plain functions that take the state and return the keys they change. **Edges**:
what runs next, fixed or chosen by a function.

ShipFast's first decision is a good fit for code: triage the message (notebook 01), and if it is out of
scope, decline without ever starting the specialist agent.

In [2]:
from typing import TypedDict

from langchain.agents import create_agent

from agent_contracts.shipfast import tools as shipfast
from agent_contracts.shipfast.schema import Triage
from lg_agents.prompts import SPECIALIST
from lg_agents.tools import shipfast_tools

TRIAGE_PROMPT = ("Classify the customer's message for ShipFast support. intent: order_status, refund_request, "
                 "complaint, or other (anything not about orders, deliveries, refunds or complaints). "
                 "order_id: the order ID such as A1001 if given, else null.")
specialist = create_agent(llm, shipfast_tools(), system_prompt=SPECIALIST)


class Support(TypedDict, total=False):
    message: str
    triage: Triage
    reply: str


def triage(state: Support) -> Support:
    return {"triage": llm.with_structured_output(Triage).invoke(
        [SystemMessage(TRIAGE_PROMPT), HumanMessage(state["message"])])}


def decline(state: Support) -> Support:  # no model call at all
    return {"reply": "Sorry, I can only help with orders, deliveries, refunds and complaints."}


def resolve(state: Support) -> Support:
    result = specialist.invoke({"messages": [HumanMessage(state["message"])]})
    return {"reply": result["messages"][-1].content}


def route(state: Support) -> str:
    return "decline" if state["triage"].intent == "other" else "resolve"


g = StateGraph(Support)
g.add_node("triage", triage)
g.add_node("decline", decline)
g.add_node("resolve", resolve)
g.add_edge(START, "triage")
g.add_conditional_edges("triage", route, ["decline", "resolve"])
g.add_edge("decline", END)
g.add_edge("resolve", END)
support = g.compile()

for e in support.get_graph().edges:
    print(f"{e.source:9} -> {e.target:8} {'(conditional)' if e.conditional else ''}")

__start__ -> triage   
triage    -> decline  (conditional)
triage    -> resolve  (conditional)
decline   -> __end__  
resolve   -> __end__  


In [3]:
for message in ("Can you write me a poem about cats?", "Where is order A9999?"):
    shipfast.reset()
    out = support.invoke({"message": message})
    print(f"{message}\n  triage: {out['triage'].intent}, {out['triage'].order_id}   tools: {shipfast.tool_calls()}")
    print(f"  reply:  {out['reply'].strip()[:150]}\n")

Can you write me a poem about cats?
  triage: other, None   tools: []
  reply:  Sorry, I can only help with orders, deliveries, refunds and complaints.



Where is order A9999?
  triage: order_status, A9999   tools: ['track_shipment']
  reply:  I'm sorry, but I couldn't find any information about order A9999. Could you please double-check the order ID or provide additional details?



The poem never reached the specialist: a fixed edge in code handled it, cheaply and predictably. The
order question went to the agent, which chose its own tools. That mix, **a workflow outside and an agent
inside**, is the pattern the crash course recommends: structure where you can, autonomy where you must.

## The research agent as a workflow

The research agent (`contracts/agent_contracts/research/SPEC.md`) must plan, research with at most 5 tool
calls, draft a cited answer, critique it, and revise at most twice. Most of that is a fixed sequence, so it
is a workflow. Only the research step is open-ended (which searches? is a calculation needed?), so that
step is a small agent. It lives in `lg_agents/research_flow.py`:

In [4]:
import lg_agents.research_flow as rf

print(rf.__doc__)
flow = rf.build_research_flow(llm)
for e in flow.get_graph().edges:
    print(f"  {e.source:9} -> {e.target:9} {'(conditional)' if e.conditional else ''}")

The research agent as a LangGraph workflow (research SPEC.md section 1), taught in notebook 05.

Workflow outside, agent inside: fixed steps (plan, draft, critique, revise) with a small tool-using agent
for the research step. The revision limit is enforced by the graph's edges, in code, never by the prompt.
The tool-call budget (5) is enforced by the shared tools.

    plan -> research -> draft -> critique --pass--> END
                                   ^   \--revise--> revise --(revisions < 2)--> critique
                                                        \--(revisions == 2)--> END

  __start__ -> plan      
  critique  -> __end__   (conditional)
  critique  -> revise    (conditional)
  draft     -> critique  
  plan      -> research  
  research  -> draft     
  revise    -> __end__   (conditional)
  revise    -> critique  (conditional)


Each node is one step, with its own instructions (`rf.RESEARCHER`, `rf.WRITER`, `rf.CRITIC`). The
writer and critic use structured output, so the draft comes back as an answer plus citations, and the
critique as a verdict plus issues, the shared `Critique` model from `agent_contracts`.

## Running it, step by step

Question Q01 asks how many employees Veltara Aquatech has. The fixtures contain the current figure (350,
in a 2026 article) and a stale one (200, in a 2023 article), and an employees search returns both. Watch
each node's update:

In [5]:
from agent_contracts.research import tools as research
from agent_contracts.research.scoring import load_questions, score_report

questions = {q["id"]: q for q in load_questions()}
q = questions["Q01_headcount"]
research.reset()
state = {"question_id": q["id"], "question": q["question"]}
for update in flow.stream(state, stream_mode="updates"):
    for node, change in update.items():
        state.update(change)
        shown = {k: (v if len(str(v)) < 160 else str(v)[:160] + " ...") for k, v in change.items()}
        print(f"[{node}] {shown}\n")

report = rf.to_report(state)
print("tool calls:", research.tool_calls())
print("score:", score_report(report, q))

[plan] {'plan': '["Search for Veltara Aquatech\'s official website to find any publicly available information about their employee count.", \'Look for recent press releases, inves ...', 'revisions': 0, 'critiques': []}



[research] {'findings': 'Tool results:\n{"query": "Veltara Aquatech employee count", "results": [{"id": "F02", "title": "Veltara grows team to 350 employees ahead of Cebu factory opening ...'}



[draft] {'answer': 'Veltara Aquatech currently has 350 employees, as reported in the article published on 2026-03-18 [F02]. This figure is the most recent available and supersedes  ...', 'citations': ['F02']}



[critique] {'critiques': [{'verdict': 'pass', 'issues': []}]}

tool calls: ['web_search']
score: {'passed': False, 'scores': {'fact_recall': 1.0, 'facts_missing': [], 'contradiction_free': False, 'citations_valid': True}, 'problems': ["repeats stale or wrong figures: ['200']"]}


Look at the saved run closely. The draft has the right figure, 350, but it adds that this "supersedes"
an older one, and in doing so mentions 200. The critic, a model call told to send back any draft that
mentions an outdated figure, passed it. The shared scorer, a plain text check, failed it
(`contradiction_free: False`). A model checking a model misses things a deterministic check does not,
which is why ShipFast and the research agent are scored in code, never by another model call.

## The loop, and its limit

When the critique's verdict is `revise`, the graph goes to `revise` and back to `critique`. The critic is
not useless: given a draft that relies on the stale figure outright, with the real findings from that run,
it does send it back:

In [6]:
stale = rf.critique_draft(llm, q["question"], state["findings"],
                          "Veltara Aquatech has 200 employees [F03].", ["F03"])
print(stale)

verdict='revise' issues=['The draft answer cites 200 employees, which is outdated. The most recent finding (F02, published 2026-03-18) states that Veltara Aquatech has 350 employees. The draft answer should reflect the latest employee count.', 'The draft answer does not include the most recent citation (F02), which is necessary for accuracy and completeness.']


What stops a critic that is never satisfied? Not the prompt. The two routing functions decide every
transition, and `route_after_revise` ends the run after the second revision whatever the critic would
say. These are plain functions, so you can check the rule directly:

In [7]:
cases = [
    ({"critiques": [{"verdict": "pass"}]}, rf.route_after_critique),
    ({"critiques": [{"verdict": "revise"}]}, rf.route_after_critique),
    ({"revisions": 1}, rf.route_after_revise),
    ({"revisions": 2}, rf.route_after_revise),
]
for state_in, rule in cases:
    print(f"{rule.__name__:22} {state_in}  ->  {rule(state_in)}")

route_after_critique   {'critiques': [{'verdict': 'pass'}]}  ->  __end__
route_after_critique   {'critiques': [{'verdict': 'revise'}]}  ->  revise
route_after_revise     {'revisions': 1}  ->  critique
route_after_revise     {'revisions': 2}  ->  __end__


So a run can only end in the three shapes the spec allows: `[pass]`, `[revise, pass]`, or
`[revise, revise]` after two revisions. `score_report` checks that shape for every report.

## Workflow or agent?

| | Workflow (`StateGraph`) | Agent (`create_agent`) |
|---|---|---|
| Who decides the next step | your code | the model |
| Predictability | high: the same path every time | lower: it can skip or repeat steps |
| Good for | known procedures, limits that must hold, cheap routing | open-ended investigation |
| In this tutorial | ShipFast routing, the research loop | ShipFast's specialist, the research step |

## How the others do this

- **CrewAI** draws workflows as a **Flow**: methods decorated `@start` and `@listen` run in order, and a
  `@router` method returns the name of the next step, like a conditional edge. Agents and crews run inside
  flow steps, the same "workflow outside, agent inside" shape.
- **Strands** builds multi-step systems from agents: `GraphBuilder` connects agents as nodes with
  conditional edges, and `Swarm` lets agents hand off to each other. Plain code steps are less central
  than in LangGraph.

## Takeaways

- A `StateGraph` is a typed state, nodes that update it, and edges, some decided by functions.
- Use code for the steps you know and an agent for the steps you do not; mix them freely.
- Put limits in the edges. A loop bounded by a routing function cannot run away.
- Next: **06 · Guardrails and approval**.